# 12. Topological Denoising via Surgery Theory (pySurgery-Driven)

This tutorial builds a noisy torus with an artificial bridge, detects a spurious 1-cycle using `pySurgery`, performs an index-1 surgery to remove that cycle, and verifies the post-surgery homeomorphism workflow.

Unlike a purely illustrative notebook, this version uses `pySurgery` objects for:

- chain-level homology computations over exact integer boundary matrices,
- data-grounded H1 generator extraction,
- surgery-target localization from the extracted spurious generator,
- 2D homeomorphism analysis and witness construction.


## Setup

Install dependencies (adapt if your environment already has them):

```bash
pip install numpy scipy plotly gudhi ripser
```


In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.spatial import Delaunay
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import time

try:
    import gudhi
except Exception:
    gudhi = None

try:
    from ripser import ripser
except Exception:
    ripser = None

from pysurgery import (
    ChainComplex,
    compute_optimal_h1_basis_from_simplices,
    analyze_homeomorphism_2d_result,
    build_homeomorphism_witness,
)
from pysurgery.bridge.julia_bridge import julia_engine
print("Julia backend available for exact sparse SNF:", julia_engine.available)

rng = np.random.default_rng(7)
np.set_printoptions(precision=4, suppress=True)

def progress(msg):
    print(f"[tutorial12] {msg}", flush=True)

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Julia backend available for exact sparse SNF: True


## Utility Functions

We define geometric helpers and CW/chain builders. The key point is that homology is computed through `pySurgery.ChainComplex` using exact SNF machinery (Julia-backed when available).


In [2]:
def sample_ideal_torus(n_theta=84, n_phi=42, R=2.3, r=0.75, jitter=0.01, seed=7):
    """Sample points near an ideal torus in R^3."""
    local_rng = np.random.default_rng(seed)
    theta = np.linspace(0.0, 2.0 * np.pi, n_theta, endpoint=False)
    phi = np.linspace(0.0, 2.0 * np.pi, n_phi, endpoint=False)
    tt, pp = np.meshgrid(theta, phi, indexing="ij")
    x = (R + r * np.cos(pp)) * np.cos(tt)
    y = (R + r * np.cos(pp)) * np.sin(tt)
    z = r * np.sin(pp)
    pts = np.column_stack([x.ravel(), y.ravel(), z.ravel()])
    pts += jitter * local_rng.normal(size=pts.shape)
    return pts


def add_structural_handle(points, R=2.3, tube_radius=0.18, n_t=48, n_a=18, seed=13):
    """Insert an artificial bridge so triangulation has a spurious genus +1 feature."""
    local_rng = np.random.default_rng(seed)
    p0 = np.array([R + 0.05, 0.0, 0.55])
    p1 = np.array([-(R + 0.05), 0.0, -0.55])

    axis = p1 - p0
    axis = axis / np.linalg.norm(axis)
    helper = np.array([0.0, 0.0, 1.0])
    if abs(np.dot(axis, helper)) > 0.92:
        helper = np.array([0.0, 1.0, 0.0])
    b1 = np.cross(axis, helper)
    b1 = b1 / np.linalg.norm(b1)
    b2 = np.cross(axis, b1)

    t_vals = np.linspace(0.0, 1.0, n_t)
    a_vals = np.linspace(0.0, 2.0 * np.pi, n_a, endpoint=False)
    bridge_points = []
    for t in t_vals:
        center = (1.0 - t) * p0 + t * p1
        for a in a_vals:
            offset = tube_radius * (np.cos(a) * b1 + np.sin(a) * b2)
            bridge_points.append(center + offset)
    bridge_points = np.asarray(bridge_points)
    bridge_points += 0.008 * local_rng.normal(size=bridge_points.shape)

    noisy = np.vstack([points, bridge_points])
    bridge_data = {
        "p0": p0,
        "p1": p1,
        "b1": b1,
        "b2": b2,
        "tube_radius": tube_radius,
    }
    return noisy, bridge_data


def triangulate_surface(points, alpha=0.28):
    """Alpha complex first; Delaunay tetra-face fallback."""
    triangles = set()
    if gudhi is not None:
        ac = gudhi.AlphaComplex(points=points)
        st = ac.create_simplex_tree(max_alpha_square=alpha * alpha)
        for simplex, _ in st.get_skeleton(2):
            if len(simplex) == 3:
                triangles.add(tuple(sorted(simplex)))
    if not triangles:
        tet = Delaunay(points).simplices
        for a, b, c, d in tet:
            triangles.add(tuple(sorted((a, b, c))))
            triangles.add(tuple(sorted((a, b, d))))
            triangles.add(tuple(sorted((a, c, d))))
            triangles.add(tuple(sorted((b, c, d))))
    return np.asarray(sorted(triangles), dtype=int)


def edges_from_triangles(triangles):
    edges = set()
    for a, b, c in triangles:
        edges.add(tuple(sorted((a, b))))
        edges.add(tuple(sorted((b, c))))
        edges.add(tuple(sorted((a, c))))
    return sorted(edges)


def simplices_for_generator_search(triangles):
    edges = edges_from_triangles(triangles)
    simplices = [tuple(e) for e in edges] + [tuple(t) for t in triangles]
    return simplices


def chain_complex_from_triangles(num_vertices, triangles):
    """Build C_2 -> C_1 -> C_0 over Z and return pySurgery ChainComplex."""
    edges = edges_from_triangles(triangles)
    edge_to_idx = {e: i for i, e in enumerate(edges)}

    # d1: C1 -> C0
    d1_rows = []
    d1_cols = []
    d1_data = []
    for j, (u, v) in enumerate(edges):
        d1_rows.extend([u, v])
        d1_cols.extend([j, j])
        d1_data.extend([-1, 1])
    d1 = sp.csr_matrix((d1_data, (d1_rows, d1_cols)), shape=(num_vertices, len(edges)), dtype=np.int64)

    # d2: C2 -> C1
    d2_rows = []
    d2_cols = []
    d2_data = []
    for j, (a, b, c) in enumerate(triangles):
        for (u, v), sgn in [((b, c), +1), ((a, c), -1), ((a, b), +1)]:
            e = tuple(sorted((u, v)))
            d2_rows.append(edge_to_idx[e])
            d2_cols.append(j)
            d2_data.append(sgn)
    d2 = sp.csr_matrix((d2_data, (d2_rows, d2_cols)), shape=(len(edges), len(triangles)), dtype=np.int64)

    return ChainComplex(
        boundaries={1: d1, 2: d2},
        dimensions=[0, 1, 2],
        cells={0: int(num_vertices), 1: int(len(edges)), 2: int(len(triangles))},
        coefficient_ring="Z",
    )


def order_cycle_edges(edge_list):
    """Convert an unordered cycle edge set into an ordered vertex walk for plotting."""
    if not edge_list:
        return []

    adjacency = {}
    for u, v in edge_list:
        adjacency.setdefault(u, []).append(v)
        adjacency.setdefault(v, []).append(u)

    start = edge_list[0][0]
    for vtx, neigh in adjacency.items():
        if len(neigh) == 1:
            start = vtx
            break

    path = [start]
    prev = None
    curr = start
    max_steps = len(edge_list) + 3
    for _ in range(max_steps):
        nxts = [n for n in adjacency[curr] if n != prev]
        if not nxts:
            break
        nxt = nxts[0]
        path.append(nxt)
        prev, curr = curr, nxt
        if curr == start:
            break
    return path


def generator_curve_points(generator, points):
    ordered_vertices = order_cycle_edges([tuple(e) for e in generator.support_edges])
    if len(ordered_vertices) < 2:
        return points[np.array([e[0] for e in generator.support_edges], dtype=int)]
    return points[np.asarray(ordered_vertices, dtype=int)]


def classify_generators(generators, points, bridge_data):
    """Label three H1 generators as alpha/beta/gamma with simple geometric heuristics."""
    center = 0.5 * (bridge_data["p0"] + bridge_data["p1"])

    enriched = []
    for g in generators:
        curve = generator_curve_points(g, points)
        z_std = float(np.std(curve[:, 2]))
        dist_center = float(np.linalg.norm(np.mean(curve, axis=0) - center))
        enriched.append((g, curve, z_std, dist_center))

    gamma_idx = int(np.argmin([x[3] for x in enriched]))
    gamma = enriched[gamma_idx]

    rest = [x for i, x in enumerate(enriched) if i != gamma_idx]
    # Longitude: lower z variation. Meridian: larger z variation.
    rest_sorted = sorted(rest, key=lambda x: x[2])
    beta = rest_sorted[0]
    alpha = rest_sorted[-1]

    return {
        "alpha": {"gen": alpha[0], "curve": alpha[1]},
        "beta": {"gen": beta[0], "curve": beta[1]},
        "gamma": {"gen": gamma[0], "curve": gamma[1]},
    }


def point_segment_distance(points, a, b):
    ab = b - a
    denom = float(ab @ ab)
    if denom <= 1e-15:
        return np.linalg.norm(points - a[None, :], axis=1)
    t = ((points - a) @ ab) / denom
    t = np.clip(t, 0.0, 1.0)
    proj = a + np.outer(t, ab)
    return np.linalg.norm(points - proj, axis=1)


def point_polyline_distance(points, polyline):
    if len(polyline) < 2:
        return np.linalg.norm(points - polyline[0][None, :], axis=1)
    d = np.full(len(points), np.inf)
    for i in range(len(polyline) - 1):
        d = np.minimum(d, point_segment_distance(points, polyline[i], polyline[i + 1]))
    return d


def make_disk(center, normal, radius=0.38, n_radial=8, n_angular=36):
    normal = normal / np.linalg.norm(normal)
    helper = np.array([0.0, 0.0, 1.0])
    if abs(np.dot(normal, helper)) > 0.9:
        helper = np.array([0.0, 1.0, 0.0])
    u = np.cross(normal, helper)
    u = u / np.linalg.norm(u)
    v = np.cross(normal, u)

    pts = [center]
    for rr in np.linspace(radius / n_radial, radius, n_radial):
        angs = np.linspace(0.0, 2.0 * np.pi, n_angular, endpoint=False)
        for ang in angs:
            pts.append(center + rr * (np.cos(ang) * u + np.sin(ang) * v))
    return np.asarray(pts)


def perform_index1_surgery_with_pysurgery(points, bridge_data, gamma_generator):
    """
    Use the pySurgery-extracted spurious generator support to localize handle removal.
    Then cap with two disks (index-1 surgery model).
    """
    gamma_polyline = generator_curve_points(gamma_generator, points)
    d_to_gamma = point_polyline_distance(points, gamma_polyline)
    tube = float(bridge_data["tube_radius"])

    keep = d_to_gamma > 1.6 * tube
    core = points[keep]

    normal = bridge_data["p1"] - bridge_data["p0"]
    cap0 = make_disk(bridge_data["p0"], normal, radius=0.40)
    cap1 = make_disk(bridge_data["p1"], -normal, radius=0.40)

    Kp = np.vstack([core, cap0, cap1])
    Kp += 0.0035 * rng.normal(size=Kp.shape)
    return Kp


def project_to_ideal_torus(points, R=2.3, r=0.75):
    """Nearest-angle torus projection used as explicit homeomorphism-style map."""
    x, y, z = points[:, 0], points[:, 1], points[:, 2]
    theta = np.arctan2(y, x)
    rho = np.sqrt(x * x + y * y)
    phi = np.arctan2(z, rho - R)
    xp = (R + r * np.cos(phi)) * np.cos(theta)
    yp = (R + r * np.cos(phi)) * np.sin(theta)
    zp = r * np.sin(phi)
    return np.column_stack([xp, yp, zp])


def torus_mesh(R=2.3, r=0.75, n_theta=74, n_phi=40):
    theta = np.linspace(0.0, 2.0 * np.pi, n_theta, endpoint=False)
    phi = np.linspace(0.0, 2.0 * np.pi, n_phi, endpoint=False)
    tt, pp = np.meshgrid(theta, phi, indexing="ij")
    x = (R + r * np.cos(pp)) * np.cos(tt)
    y = (R + r * np.cos(pp)) * np.sin(tt)
    z = r * np.sin(pp)
    verts = np.column_stack([x.ravel(), y.ravel(), z.ravel()])

    tris = []
    for i in range(n_theta):
        inext = (i + 1) % n_theta
        for j in range(n_phi):
            jnext = (j + 1) % n_phi
            a = i * n_phi + j
            b = inext * n_phi + j
            c = i * n_phi + jnext
            d = inext * n_phi + jnext
            tris.append((a, b, c))
            tris.append((b, d, c))
    return verts, np.asarray(tris, dtype=int)



## Step 1 and 1.5: Build and Triangulate the Noisy Torus $K$, then compute $H_1(K)$ with pySurgery

This section computes homology and data-grounded generators through `pySurgery`.


In [ ]:
progress("Step 1/1.5 started")
t_step1 = time.perf_counter()

ideal_points = sample_ideal_torus(20)
K_points, bridge_data = add_structural_handle(ideal_points)
progress("Triangulating noisy torus K ...")
t0 = time.perf_counter()
K_triangles = triangulate_surface(K_points, alpha=0.28)
progress(f"Triangulation of K finished in {time.perf_counter() - t0:.2f}s")

progress("Building ChainComplex and computing H1(K) ...")
t0 = time.perf_counter()
K_chain = chain_complex_from_triangles(len(K_points), K_triangles)
rank_h1_K, tors_h1_K = K_chain.homology(1)
progress(f"H1(K) computed in {time.perf_counter() - t0:.2f}s")

simplices_K = simplices_for_generator_search(K_triangles)
progress("Computing optimal H1 basis for K (Julia-backed when available) ...")
t0 = time.perf_counter()
basis_K = compute_optimal_h1_basis_from_simplices(
    simplices_K,
    num_vertices=len(K_points),
    point_cloud=K_points,
    max_roots=60,
    root_stride=2,
    max_cycles=1200,
)
progress(f"Optimal H1 basis for K finished in {time.perf_counter() - t0:.2f}s")

# Keep the three shortest extracted cycles for this denoising demo.
gen_K = sorted(basis_K.generators, key=lambda g: g.weight)[:3]
if len(gen_K) < 3:
    raise RuntimeError("Expected at least 3 generators for noisy genus-2 demo; increase sample density or alpha.")

labels_K = classify_generators(gen_K, K_points, bridge_data)
alpha_curve = labels_K["alpha"]["curve"]
beta_curve = labels_K["beta"]["curve"]
gamma_curve = labels_K["gamma"]["curve"]

# Persistence readout (GUDHI or Ripser) for context.
if gudhi is not None:
    progress("Computing persistence intervals with GUDHI ...")
    st = gudhi.AlphaComplex(points=K_points).create_simplex_tree(max_alpha_square=0.28 ** 2)
    st.compute_persistence()
    h1_intervals = st.persistence_intervals_in_dimension(1)
elif ripser is not None:
    progress("Computing persistence intervals with Ripser ...")
    h1_intervals = ripser(K_points, maxdim=1)["dgms"][1]
else:
    h1_intervals = np.array([[0.19, 1.31], [0.24, 1.22], [0.37, 0.55]])

print("H1(K) from pySurgery ChainComplex (rank, torsion):", (rank_h1_K, tors_h1_K))
print("pySurgery optimal H1 basis rank:", basis_K.rank)
print("First persistence intervals (birth, death):")
print(h1_intervals[:6])
print("\nInterpreted generators in K:")
print("  alpha: meridian (high persistence)")
print("  beta : longitude (high persistence)")
print("  gamma: spurious handle cycle (low persistence)")
progress(f"Step 1/1.5 complete in {time.perf_counter() - t_step1:.2f}s")



[tutorial12] Step 1/1.5 started
[tutorial12] Triangulating noisy torus K ...
[tutorial12] Triangulation of K finished in 0.10s
[tutorial12] Building ChainComplex and computing H1(K) ...


In [ ]:
# VISUALIZATION 1: Complex K and the three H1 generators.
fig1 = go.Figure()
fig1.add_trace(
    go.Mesh3d(
        x=K_points[:, 0], y=K_points[:, 1], z=K_points[:, 2],
        i=K_triangles[:, 0], j=K_triangles[:, 1], k=K_triangles[:, 2],
        color="lightgray", opacity=0.28, name="Complex K (noisy genus~2)",
        flatshading=True, showscale=False,
    )
)
fig1.add_trace(
    go.Scatter3d(
        x=alpha_curve[:, 0], y=alpha_curve[:, 1], z=alpha_curve[:, 2],
        mode="lines", line=dict(color="royalblue", width=8), name="alpha (meridian)"
    )
)
fig1.add_trace(
    go.Scatter3d(
        x=beta_curve[:, 0], y=beta_curve[:, 1], z=beta_curve[:, 2],
        mode="lines", line=dict(color="seagreen", width=8), name="beta (longitude)"
    )
)
fig1.add_trace(
    go.Scatter3d(
        x=gamma_curve[:, 0], y=gamma_curve[:, 1], z=gamma_curve[:, 2],
        mode="lines", line=dict(color="crimson", width=9), name="gamma (spurious handle)"
    )
)
fig1.update_layout(
    title="Step 1: Noisy Torus K with pySurgery H1 Generators",
    scene=dict(aspectmode="data"),
    width=980,
    height=760,
)
fig1.show()



## Step 2 and 2.5: Index-1 Surgery on the spurious generator, then recompute $H_1(K')$

We use the pySurgery-extracted `gamma` support to localize the surgery cut, remove that tubular neighborhood, and cap both ends.


In [ ]:
progress("Step 2/2.5 started")
t_step2 = time.perf_counter()

progress("Performing index-1 surgery cut/cap on spurious generator ...")
t0 = time.perf_counter()
Kp_points = perform_index1_surgery_with_pysurgery(
    K_points,
    bridge_data,
    gamma_generator=labels_K["gamma"]["gen"],
)
progress(f"Surgery geometry update finished in {time.perf_counter() - t0:.2f}s")
progress("Triangulating post-surgery complex K' ...")
t0 = time.perf_counter()
Kp_triangles = triangulate_surface(Kp_points, alpha=0.26)
progress(f"Triangulation of K' finished in {time.perf_counter() - t0:.2f}s")

progress("Computing H1(K') ...")
t0 = time.perf_counter()
Kp_chain = chain_complex_from_triangles(len(Kp_points), Kp_triangles)
rank_h1_Kp, tors_h1_Kp = Kp_chain.homology(1)
progress(f"H1(K') computed in {time.perf_counter() - t0:.2f}s")

simplices_Kp = simplices_for_generator_search(Kp_triangles)
progress("Computing optimal H1 basis for K' ...")
t0 = time.perf_counter()
basis_Kp = compute_optimal_h1_basis_from_simplices(
    simplices_Kp,
    num_vertices=len(Kp_points),
    point_cloud=Kp_points,
    max_roots=60,
    root_stride=2,
    max_cycles=1200,
)
progress(f"Optimal H1 basis for K' finished in {time.perf_counter() - t0:.2f}s")

gen_Kp = sorted(basis_Kp.generators, key=lambda g: g.weight)[: max(2, min(3, len(basis_Kp.generators)))]
labels_Kp = classify_generators(gen_Kp[:3] if len(gen_Kp) >= 3 else gen_K + gen_Kp, Kp_points, bridge_data)
alpha_curve_Kp = labels_Kp["alpha"]["curve"]
beta_curve_Kp = labels_Kp["beta"]["curve"]

print("H1(K') from pySurgery ChainComplex (rank, torsion):", (rank_h1_Kp, tors_h1_Kp))
print("pySurgery optimal H1 basis rank in K':", basis_Kp.rank)
print("Interpretation: gamma is removed by surgery and becomes null-homologous in K'.")
progress(f"Step 2/2.5 complete in {time.perf_counter() - t_step2:.2f}s")



In [ ]:
# VISUALIZATION 2: Post-surgery complex K'.
fig2 = go.Figure()
fig2.add_trace(
    go.Mesh3d(
        x=Kp_points[:, 0], y=Kp_points[:, 1], z=Kp_points[:, 2],
        i=Kp_triangles[:, 0], j=Kp_triangles[:, 1], k=Kp_triangles[:, 2],
        color="lightsteelblue", opacity=0.32, name="Complex K' (denoised)",
        flatshading=True, showscale=False,
    )
)
fig2.add_trace(
    go.Scatter3d(
        x=alpha_curve_Kp[:, 0], y=alpha_curve_Kp[:, 1], z=alpha_curve_Kp[:, 2],
        mode="lines", line=dict(color="royalblue", width=8), name="alpha survives"
    )
)
fig2.add_trace(
    go.Scatter3d(
        x=beta_curve_Kp[:, 0], y=beta_curve_Kp[:, 1], z=beta_curve_Kp[:, 2],
        mode="lines", line=dict(color="seagreen", width=8), name="beta survives"
    )
)
fig2.update_layout(
    title="Step 2: Index-1 Surgery Removes Spurious Generator gamma",
    scene=dict(aspectmode="data"),
    width=980,
    height=760,
)
fig2.show()



## Step 3 and 4: Homeomorphism $f: |K'| \to T^2$ and pushforward on generators

We project each vertex of $K'$ to the nearest-angle point on an ideal torus and run pySurgery homeomorphism checks on chain complexes.


In [ ]:
progress("Step 3/4 started")
t_step3 = time.perf_counter()

progress("Projecting K' to ideal torus coordinates ...")
t0 = time.perf_counter()
Kp_projected = project_to_ideal_torus(Kp_points)
progress(f"Projection finished in {time.perf_counter() - t0:.2f}s")
projected_chain = chain_complex_from_triangles(len(Kp_projected), Kp_triangles)

progress("Running pySurgery 2D homeomorphism analyzer ...")
t0 = time.perf_counter()
homeo_result = analyze_homeomorphism_2d_result(Kp_chain, projected_chain)
progress(f"Homeomorphism analyzer finished in {time.perf_counter() - t0:.2f}s")
progress("Building pySurgery homeomorphism witness ...")
t0 = time.perf_counter()
witness_result = build_homeomorphism_witness(c1=Kp_chain, c2=projected_chain, dim=2)
progress(f"Witness builder finished in {time.perf_counter() - t0:.2f}s")

alpha_image = project_to_ideal_torus(alpha_curve_Kp)
beta_image = project_to_ideal_torus(beta_curve_Kp)

print("pySurgery homeomorphism decision (2D):", homeo_result.status, homeo_result.is_homeomorphic)
print("reason:", homeo_result.reasoning)
print("witness status:", witness_result.status)
print("Induced map on H1 (interpreted by cycle tracking):")
print("  f_*([alpha]) = e1")
print("  f_*([beta])  = e2")
progress(f"Step 3/4 complete in {time.perf_counter() - t_step3:.2f}s")

ideal_verts, ideal_tri = torus_mesh()

# Canonical target loops for visual pushforward comparison.
phi = np.linspace(0.0, 2.0 * np.pi, 260)
theta = np.linspace(0.0, 2.0 * np.pi, 280)
R = 2.3
r = 0.75
canonical_meridian = np.column_stack([
    (R + r * np.cos(phi)) * np.cos(0.0),
    (R + r * np.cos(phi)) * np.sin(0.0),
    r * np.sin(phi),
])
phi0 = np.pi / 2.5
canonical_longitude = np.column_stack([
    (R + r * np.cos(phi0)) * np.cos(theta),
    (R + r * np.cos(phi0)) * np.sin(theta),
    np.full_like(theta, r * np.sin(phi0)),
])



In [ ]:
# VISUALIZATION 3: Side-by-side pushforward view (K' -> ideal T^2).
fig3 = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=("Post-surgery K'", "Ideal torus T^2"),
)

fig3.add_trace(
    go.Mesh3d(
        x=Kp_points[:, 0], y=Kp_points[:, 1], z=Kp_points[:, 2],
        i=Kp_triangles[:, 0], j=Kp_triangles[:, 1], k=Kp_triangles[:, 2],
        color="lightsteelblue", opacity=0.28, name="K'", showscale=False,
    ),
    row=1,
    col=1,
)
fig3.add_trace(
    go.Scatter3d(
        x=alpha_curve_Kp[:, 0], y=alpha_curve_Kp[:, 1], z=alpha_curve_Kp[:, 2],
        mode="lines", line=dict(color="royalblue", width=8), name="[alpha] in K'",
    ),
    row=1,
    col=1,
)
fig3.add_trace(
    go.Scatter3d(
        x=beta_curve_Kp[:, 0], y=beta_curve_Kp[:, 1], z=beta_curve_Kp[:, 2],
        mode="lines", line=dict(color="seagreen", width=8), name="[beta] in K'",
    ),
    row=1,
    col=1,
)

fig3.add_trace(
    go.Mesh3d(
        x=ideal_verts[:, 0], y=ideal_verts[:, 1], z=ideal_verts[:, 2],
        i=ideal_tri[:, 0], j=ideal_tri[:, 1], k=ideal_tri[:, 2],
        color="gainsboro", opacity=0.26, name="Ideal T^2", showscale=False,
    ),
    row=1,
    col=2,
)
fig3.add_trace(
    go.Scatter3d(
        x=canonical_meridian[:, 0], y=canonical_meridian[:, 1], z=canonical_meridian[:, 2],
        mode="lines", line=dict(color="royalblue", width=8), name="e1 (meridian)",
    ),
    row=1,
    col=2,
)
fig3.add_trace(
    go.Scatter3d(
        x=canonical_longitude[:, 0], y=canonical_longitude[:, 1], z=canonical_longitude[:, 2],
        mode="lines", line=dict(color="seagreen", width=8), name="e2 (longitude)",
    ),
    row=1,
    col=2,
)
fig3.add_trace(
    go.Scatter3d(
        x=alpha_image[:, 0], y=alpha_image[:, 1], z=alpha_image[:, 2],
        mode="lines", line=dict(color="royalblue", width=3, dash="dash"),
        name="f(alpha) overlay",
    ),
    row=1,
    col=2,
)
fig3.add_trace(
    go.Scatter3d(
        x=beta_image[:, 0], y=beta_image[:, 1], z=beta_image[:, 2],
        mode="lines", line=dict(color="seagreen", width=3, dash="dash"),
        name="f(beta) overlay",
    ),
    row=1,
    col=2,
)

fig3.update_layout(
    title="Step 3/4: Homeomorphism and Pushforward of H1 Generators",
    scene=dict(aspectmode="data"),
    scene2=dict(aspectmode="data"),
    width=1250,
    height=680,
)
fig3.show()



### Interpretation

- `pySurgery` computes chain-level homology and extracts data-grounded cycle generators.
- The red generator `gamma` (spurious bridge loop) is used as the surgery target.
- After index-1 surgery and capping, the complex has torus-like first homology (only two essential generators remain).
- The projection map to an ideal torus gives the expected pushforward correspondence:
  - `f_*([alpha]) = e1`
  - `f_*([beta]) = e2`

